[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [3]:
import torch
import math

In [1]:
import math
import torch
from torch import Tensor


def flash_attention(Q: Tensor, K: Tensor, V: Tensor, block_size: int = 32) -> Tensor:
    """
    Tiled Flash Attention with online softmax.

    Args:
        Q: (B, S, D)
        K: (B, S, D)
        V: (B, S, D)
        block_size: int

    Returns:
        O: (B, S, D)

    Computes:
        O = softmax(Q K^T / sqrt(D)) V

    Key idea:
        Do not materialize full attention matrix of shape (B, S, S).
        Instead, process Q in blocks and K/V in blocks.
    """

    B, S, D = Q.shape
    scale = 1.0 / math.sqrt(D)

    # Output buffer
    # O: (B, S, D)
    O = torch.empty_like(Q)

    # Loop over Q blocks
    for q_start in range(0, S, block_size):
        q_end = min(q_start + block_size, S)

        # Q_block: (B, Bq, D)
        # Bq <= block_size
        Q_block = Q[:, q_start:q_end, :]
        Bq = q_end - q_start

        # For each query row, maintain online softmax statistics.

        # m: running max for each query row
        # m shape: (B, Bq)
        m = torch.full(
            (B, Bq),
            -float("inf"),
            device=Q.device,
            dtype=Q.dtype,
        )

        # l: running denominator of softmax
        # l[b, i] = sum_j exp(score[b, i, j] - m[b, i])
        # l shape: (B, Bq)
        l = torch.zeros(
            (B, Bq),
            device=Q.device,
            dtype=Q.dtype,
        )

        # acc: running numerator of attention
        # acc[b, i, :] = sum_j exp(score[b, i, j] - m[b, i]) * V[b, j, :]
        # acc shape: (B, Bq, D)
        acc = torch.zeros(
            (B, Bq, D),
            device=Q.device,
            dtype=Q.dtype,
        )

        # Loop over K/V blocks
        for k_start in range(0, S, block_size):
            k_end = min(k_start + block_size, S)

            # K_block: (B, Bk, D)
            # V_block: (B, Bk, D)
            # Bk <= block_size
            K_block = K[:, k_start:k_end, :]
            V_block = V[:, k_start:k_end, :]
            Bk = k_end - k_start

            # Compute score block.
            #
            # Q_block:             (B, Bq, D)
            # K_block.transpose:   (B, D, Bk)
            # scores:              (B, Bq, Bk)
            scores = torch.matmul(
                Q_block,
                K_block.transpose(-1, -2),
            ) * scale

            # block_m is the max score inside this K block, per query row.
            #
            # scores:   (B, Bq, Bk)
            # block_m:  (B, Bq)
            block_m = scores.max(dim=-1).values

            # New running max after seeing this block.
            #
            # m:       (B, Bq)
            # block_m: (B, Bq)
            # new_m:   (B, Bq)
            new_m = torch.maximum(m, block_m)

            # Rescale old statistics from old max m to new max new_m.
            #
            # old_scale[b, i] = exp(m_old[b, i] - m_new[b, i])
            #
            # m:         (B, Bq)
            # new_m:     (B, Bq)
            # old_scale: (B, Bq)
            old_scale = torch.exp(m - new_m)

            # Compute exp scores for current block using new_m as reference.
            #
            # scores:              (B, Bq, Bk)
            # new_m.unsqueeze(-1): (B, Bq, 1)
            # exp_scores:          (B, Bq, Bk)
            exp_scores = torch.exp(scores - new_m.unsqueeze(-1))

            # Current block's softmax denominator contribution.
            #
            # exp_scores: (B, Bq, Bk)
            # block_l:   (B, Bq)
            block_l = exp_scores.sum(dim=-1)

            # Current block's weighted value contribution.
            #
            # exp_scores: (B, Bq, Bk)
            # V_block:    (B, Bk, D)
            # block_acc:  (B, Bq, D)
            block_acc = torch.matmul(exp_scores, V_block)

            # Online update:
            #
            # Old acc was based on old max m.
            # Convert it to new max new_m by multiplying old_scale.
            #
            # acc:                     (B, Bq, D)
            # old_scale.unsqueeze(-1): (B, Bq, 1)
            # block_acc:               (B, Bq, D)
            # updated acc:             (B, Bq, D)
            acc = acc * old_scale.unsqueeze(-1) + block_acc

            # Same update for denominator.
            #
            # l:         (B, Bq)
            # old_scale: (B, Bq)
            # block_l:   (B, Bq)
            # updated l: (B, Bq)
            l = l * old_scale + block_l

            # Update running max.
            #
            # m: (B, Bq)
            m = new_m

        # Normalize accumulated numerator by accumulated denominator.
        #
        # acc:             (B, Bq, D)
        # l.unsqueeze(-1): (B, Bq, 1)
        # O_block:         (B, Bq, D)
        O_block = acc / l.unsqueeze(-1)

        # Write this Q block into output.
        #
        # O[:, q_start:q_end, :]: (B, Bq, D)
        O[:, q_start:q_end, :] = O_block

    return O

In [4]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (24.9ms)
  ✅ [2/4] Non-aligned block size (2.8ms)
  ✅ [3/4] Block size invariant (2.8ms)
  ✅ [4/4] Gradient flow (37.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (68.4ms total)
  Progress saved. Run status() to see your dashboard.

